# 09. Connecting Python to Jira

The **`jira`** package is a Python client for the Jira REST API. Instead of building HTTP requests by hand, you call methods such as `jira.issue("MYP-8")` and read fields as attributes.

Package page: https://pypi.org/project/jira/

> **Before you run this notebook:** create an API token (topic 08) and set `JIRA_SERVER`, `JIRA_EMAIL` and `JIRA_API_TOKEN` as environment variables, or in Colab's **Secrets** panel. The outputs shown were recorded on a practice Jira Cloud site with a project called **myproject** (key `MYP`). Your issue keys and data will differ.

## Install the Library

In [1]:
!pip install -U jira

> Use a current version (3.10 or newer). Atlassian retired the old Jira Cloud search endpoint, and recent versions of the library switch to the new one automatically.

On your own machine, run `pip install -U jira` (Windows) or `pip3 install -U jira` (Linux) in a terminal instead.

## Why a Password Fails

The first attempt most people make is to log in with their account password:

```python
jira = JIRA("https://your-domain.atlassian.net", basic_auth=("you@example.com", "your-password"))
```

Jira Cloud rejects this with an authentication error, because basic authentication with passwords is disabled for the REST API. **Use an API token instead** (topic 08). The token replaces the password in the same `basic_auth` pair.

## Connect with an API Token

In [2]:
import os
from jira import JIRA

jira = JIRA(
    server=os.environ["JIRA_SERVER"],                 # https://your-domain.atlassian.net
    basic_auth=(os.environ["JIRA_EMAIL"], os.environ["JIRA_API_TOKEN"]),
)

- `server` is your site URL, without anything after `.net`.
- `basic_auth` is a tuple of your Atlassian **email** and the **API token**.
- Nothing is printed if the connection works. A wrong token raises `JIRAError: HTTP 401`.

## Fetch One Issue

In [3]:
ticket = "MYP-8"
issue = jira.issue(ticket)
print(type(issue))

<class 'jira.resources.Issue'>


`jira.issue()` returns an **Issue** object. Its data lives under `issue.fields`, following the JSON structure shown in topic 08.

## Reading Fields

In [4]:
print(issue.fields.project)

MYP


In [5]:
print(issue.fields.project.key)

MYP


In [6]:
print(issue.fields.project.name)

myproject


In [7]:
print(issue.fields.issuetype.name)

Bug


In [8]:
print(issue.fields.reporter.displayName)

Ahmad Hassan


In [9]:
print(issue.fields.summary)

Critical Bug


`issue.fields.project` prints the key because the library's objects print a short, readable form. Use `.key` or `.name` when you need a specific value.

## More Fields from Another Issue

These outputs were recorded on a second practice issue, `AC-2`, in the `cloudops_cloud` project:

```python
>>> issue = jira.issue("AC-2")
>>> print(issue.fields.summary)
We have issue in CloudOps cloud and we found some links issue
>>> print(issue.fields.description)
None
>>> print(issue.fields.assignee)
DevOps Engineer
>>> print(issue.fields.status)
To Do
>>> print(issue.fields.status.name)
To Do
>>> print(issue.fields.priority)
Medium
>>> print(issue.fields.timeoriginalestimate)
18000
>>> print(issue.fields.timetracking.originalEstimate)
5h
>>> print(issue.fields.timetracking.remainingEstimate)
3h
>>> print(issue.fields.timetracking.timeSpent)
2h
```

Notes:

- `description` is `None` when the issue has no description, so check before using it.
- `timeoriginalestimate` is in **seconds** (18000 s = 5 h). `timetracking` gives the same values as readable text.

### Issue Objects Are Snapshots

An Issue object holds the data from the moment you fetched it. If someone edits the summary in the browser, your old object doesn't change. Fetch it again:

```python
issue = jira.issue("AC-2")      # fetch again to see the latest summary
print(issue.fields.summary)
```

## Listing Projects and Fields

In [10]:
for project in jira.projects():
    print(project.key, project.name)

In [11]:
# every field on the site, including custom fields (customfield_10016 and so on)
for field in jira.fields()[:10]:
    print(field["id"], "|", field["name"])

`jira.fields()` is how you find the ID of a custom field, such as story points, before reading or setting it in a script.

## Practice

1. Fetch an issue from your own project and print its key, summary, status and priority on one line.
2. Print `"No description"` when `issue.fields.description` is empty, otherwise the description.
3. Print the original estimate in hours, calculated from `timeoriginalestimate`.

**Solution:**

In [12]:
issue = jira.issue("MYP-8")
print(issue.key, "|", issue.fields.summary, "|", issue.fields.status.name, "|", issue.fields.priority.name)

print(issue.fields.description or "No description")

seconds = issue.fields.timeoriginalestimate
print(f"Estimate: {seconds / 3600:.1f} hours" if seconds else "No estimate")